<a href="https://colab.research.google.com/github/mdsadaqathali/week3/blob/main/w3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
df = pd.read_csv("Mall_Customers[1].csv")
print(df.head())
print(df.shape)
print(df.columns)
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(df["Gender"].value_counts())
plt.figure(figsize=(8,6))
sns.scatterplot(data=df,x="Annual Income (k$)",y="Spending Score (1-100)",hue="Gender")
plt.title("Annual Income vs Spending Score")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.show()
df["Gender_Encoded"] = df["Gender"].map({"Male":0,"Female":1})
print("\nENCODED DATA")
print(df.head())
features = df[["Annual Income (k$)","Spending Score (1-100)"]]
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
print("\nSCALED FEATURES")
print(scaled_features[:5])
inertia = []
silhouette_scores = []
k_values = range(2,11)
for k in k_values:
    model = KMeans(n_clusters=k,random_state=42,n_init=10)
    labels = model.fit_predict(scaled_features)
    inertia.append(model.inertia_)
    silhouette_scores.append(silhouette_score(scaled_features,labels))
plt.figure(figsize=(8,5))
plt.plot(range(2,11),inertia,marker="o")
plt.title("Elbow Method")
plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.show()
plt.figure(figsize=(8,5))
plt.plot(range(2,11),silhouette_scores,marker="o")
plt.title("Silhouette Score")
plt.xlabel("Number of Clusters")
plt.ylabel("Silhouette Score")
plt.show()
print("\nSILHOUETTE SCORES")
for k,score in zip(k_values,silhouette_scores):
    print("K =",k,"Score =",round(score,4))
kmeans = KMeans(n_clusters=5,random_state=42,n_init=10)
df["Cluster"] = kmeans.fit_predict(scaled_features)
print("\nCLUSTER COUNTS")
print(df["Cluster"].value_counts().sort_index())
centroids_scaled = kmeans.cluster_centers_
centroids = scaler.inverse_transform(centroids_scaled)
centroid_df = pd.DataFrame(centroids,columns=["Annual Income (k$)","Spending Score (1-100)"])
print("\nCLUSTER CENTROIDS")
print(centroid_df)
plt.figure(figsize=(9,7))
plt.scatter(scaled_features[:,0],scaled_features[:,1],c=df["Cluster"],cmap="viridis",s=60)
plt.scatter(centroids_scaled[:,0],centroids_scaled[:,1],marker="*",s=300)
plt.title("Customer Segmentation using K-Means")
plt.xlabel("Annual Income (Scaled)")
plt.ylabel("Spending Score (Scaled)")
plt.show()
cluster_summary = df.groupby("Cluster")[["Age","Annual Income (k$)","Spending Score (1-100)"]].mean()
print("\nCLUSTER SUMMARY")
print(cluster_summary)
for cluster in range(5):
    income = cluster_summary.loc[cluster,"Annual Income (k$)"]
    spending = cluster_summary.loc[cluster,"Spending Score (1-100)"]
    if income >= 60 and spending >= 60:
        name = "High Income High Spenders"
    elif income >= 60 and spending < 60:
        name = "High Income Low Spenders"
    elif income < 60 and spending >= 60:
        name = "Budget Conscious"
    else:
        name = "Low Income Low Spenders"
    print("Cluster",cluster,":",name)
print("\nBUSINESS REPORT")
print("High Income High Spenders: Premium products, loyalty rewards and exclusive offers can be recommended.")
print("High Income Low Spenders: Personalized offers and product recommendations can encourage more spending.")
print("Budget Conscious: Discounts, value offers and affordable products can be recommended.")
print("Low Income Low Spenders: Budget products, basic offers and low-cost promotions can be recommended.")
print("\nFINAL DATA")
print(df.head(10))
df.to_csv("customer_segments.csv",index=False)
print("Customer segmentation data saved as customer_segments.csv")